## Environment, Toggles, Imports

In [1]:
CFG = dict(
    SEED          = 42,
    N_FOLDS       = 2,
    EPOCHS_LSTM   = 2,
    EPOCHS_DEB    = 3,
    EPOCHS_QWEN   = 2,
    BS_LSTM       = 12,
    BS_DEB        = 2,
    BS_QWEN       = 1,
    GRAD_ACCUM_DEB = 4,
    GRAD_ACCUM_QWEN= 8,
    MAX_LEN_DEB   = 176,
    MAX_LEN_QWEN  = 512,
    MAX_LEN_RM    = 384,
    LR_LSTM       = 1e-3,
    LR_DEB        = 1e-5,
    LR_QWEN       = 1e-4,
    LSTM_HIDDEN   = 256,
    LSTM_EMBED    = 200,
    MARGIN        = 0.5,

    RUN_LSTM      = True,
    RUN_DEBERTA   = True,
    RUN_QWEN      = True,   
    RUN_REWARD    = True,
    RUN_RAG       = False,
    RUN_SYNTH_GEN = False, 
    USE_WANDB     = False,  

    DATA_DIR      = '/kaggle/input/competitions/smart-mcq-solver-challenge'
)

In [2]:
import os

CFG["DATA_DIR"] = "/kaggle/input/competitions/smart-mcq-solver-challenge"
CFG["OUTPUT_DIR"] = "/kaggle/working"

os.makedirs(CFG["OUTPUT_DIR"], exist_ok=True)

print(f"DATA_DIR = {CFG['DATA_DIR']}")
print(f"OUTPUT_DIR = {CFG['OUTPUT_DIR']}")

DATA_DIR = /kaggle/input/competitions/smart-mcq-solver-challenge
OUTPUT_DIR = /kaggle/working


In [3]:
import os, sys, json, math, random, time, gc, warnings, re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler

from sklearn.model_selection import KFold
from sklearn.metrics import f1_score, accuracy_score
import lightgbm as lgb

warnings.filterwarnings('ignore')

# Seeding
def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
seed_all(CFG['SEED'])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB")


Device: cpu


## Data Loading

In [4]:
train = pd.read_csv(f"{CFG['DATA_DIR']}/train.csv")
test  = pd.read_csv(f"{CFG['DATA_DIR']}/test.csv")
ss    = pd.read_csv(f"{CFG['DATA_DIR']}/sample_submission.csv")
OPTS  = ['A','B','C','D','E']

print(f"train: {train.shape} | test: {test.shape} | sample sub: {ss.shape}")
print(f"missing in train: {train.isna().sum().sum()}, in test: {test.isna().sum().sum()}")
print(f"unique answers: {sorted(train['answer'].unique())}")
print(f"answer distribution:\n{train['answer'].value_counts().reindex(OPTS)}")


train: (2000, 8) | test: (500, 7) | sample sub: (500, 2)
missing in train: 0, in test: 0
unique answers: ['A', 'B', 'C', 'D', 'E']
answer distribution:
answer
A    369
B    490
C    459
D    358
E    324
Name: count, dtype: int64


## RAG with e5-large-v2 + FAISS

In [5]:
try:
    import faiss
    HAS_FAISS = True
except ImportError:
    HAS_FAISS = False
    print('faiss not available — using sklearn NearestNeighbors fallback.')

def build_rag_context(train_df, test_df, top_k=3, max_chunks=200_000, max_chars=400):
   
    if not CFG['RUN_RAG']:
        return {}

    # 1) Build corpus
    wiki_path = Path(CFG['WIKI_DIR'])
    corpus_chunks = []
    if wiki_path.is_dir():
     
        pq_files = list(wiki_path.rglob('*.parquet'))[:8] 
        print(f'Loading {len(pq_files)} Wikipedia parquet files for RAG corpus...')
        for f in pq_files:
            try:
                df_w = pd.read_parquet(f, columns=['text'])
                texts = df_w['text'].dropna().tolist()
                for t in texts:
                    # Split into ~400-char chunks
                    for i in range(0, min(len(t), 4000), max_chars):
                        chunk = t[i:i+max_chars]
                        if len(chunk) > 80:
                            corpus_chunks.append(chunk)
                            if len(corpus_chunks) >= max_chunks:
                                break
                if len(corpus_chunks) >= max_chunks:
                    break
            except Exception as e:
                print(f'  skip {f.name}: {e}')
        print(f'  Wikipedia corpus: {len(corpus_chunks):,} chunks')
    else:
        print(f'No Wikipedia at {wiki_path}; using self-retrieval (option pool).')
     
        for _, r in train_df.iterrows():
            for c in OPTS:
                corpus_chunks.append(str(r[c]))

    if not corpus_chunks:
        return {}


    try:
        from sentence_transformers import SentenceTransformer
        print(f'Loading {CFG["EMBED_MODEL"]}...')
        encoder = SentenceTransformer(CFG['EMBED_MODEL'], device=str(DEVICE))
    except Exception as e:
        print(f'Could not load sentence encoder ({e}); skipping RAG.')
        return {}

    # E5 requires the "passage: " / "query: " prefix
    passages = [f'passage: {c}' for c in corpus_chunks]
    print('Encoding corpus...')
    corpus_emb = encoder.encode(passages, batch_size=64, show_progress_bar=True,
                                 convert_to_numpy=True, normalize_embeddings=True)

    queries = [f'query: {p}' for p in pd.concat([train_df['prompt_n'], test_df['prompt_n']]).tolist()]
    all_ids = pd.concat([train_df['id'], test_df['id']]).values
    print('Encoding queries...')
    q_emb = encoder.encode(queries, batch_size=64, show_progress_bar=True,
                            convert_to_numpy=True, normalize_embeddings=True)


    if HAS_FAISS:
        index = faiss.IndexFlatIP(corpus_emb.shape[1])
        index.add(corpus_emb.astype('float32'))
        _, I = index.search(q_emb.astype('float32'), top_k)
    else:
        from sklearn.neighbors import NearestNeighbors
        nn = NearestNeighbors(n_neighbors=top_k, metric='cosine').fit(corpus_emb)
        _, I = nn.kneighbors(q_emb)

    context_map = {}
    for qi, qid in enumerate(all_ids):
        ctx = ' '.join(corpus_chunks[ci] for ci in I[qi])
        context_map[int(qid)] = ctx[:1500]

    # Free GPU memory
    del encoder, corpus_emb, q_emb
    gc.collect(); torch.cuda.empty_cache()
    return context_map

CONTEXT_MAP = build_rag_context(train, test, top_k=3) if CFG['RUN_RAG'] else {}
print(f'RAG context built for {len(CONTEXT_MAP)} questions')


faiss not available — using sklearn NearestNeighbors fallback.
RAG context built for 0 questions


## Hard-Negative Synthetic Generation

In [6]:
def hard_negative_augment(df, n_per_row=1, sim_top_k=20, seed=0):

    if not CFG['RUN_SYNTH_GEN']:
        return pd.DataFrame()
    from sklearn.feature_extraction.text import TfidfVectorizer
    rng = np.random.default_rng(seed)

    correct_texts = df.apply(lambda r: r[r['answer']], axis=1).tolist()
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(correct_texts)
    M = vec.transform(correct_texts)
    sims = (M @ M.T).toarray()
    np.fill_diagonal(sims, -1)

    new_rows = []
    for i, (_, r) in enumerate(df.iterrows()):
        correct = r['answer']
        correct_text = r[correct]
        # top-k most-similar OTHER correct answers
        top_idx = sims[i].argsort()[::-1][:sim_top_k]
        for _ in range(n_per_row):
            chosen = rng.choice(top_idx, size=4, replace=False)
            distractors = [correct_texts[ci] for ci in chosen]
            new_opts = distractors[:]
            ans_pos = rng.integers(0, 5)
            new_opts.insert(ans_pos, correct_text)
            new_row = {'id': 100_000_000 + len(new_rows),
                       'prompt': r['prompt'], 'answer': OPTS[ans_pos]}
            for k, o in enumerate(OPTS):
                new_row[o] = new_opts[k]
            new_rows.append(new_row)

    aug = pd.DataFrame(new_rows)
    if not aug.empty:
        aug['prompt_n'] = aug['prompt'].map(normalize_text)
        for o in OPTS:
            aug[f'{o}_n'] = aug[o].map(normalize_text)
    return aug

# Also keep a simple position-shuffle augmentation as a cheap baseline
def shuffle_augment(df, n_per_row=1, seed=0):
    rng = np.random.default_rng(seed)
    rows = []
    for _, r in df.iterrows():
        for _ in range(n_per_row):
            perm = rng.permutation(5)
            opts_orig = [r[c] for c in OPTS]
            ans_idx = OPTS.index(r['answer'])
            new = {'id': 200_000_000 + len(rows), 'prompt': r['prompt']}
            for new_k, old_k in enumerate(perm):
                new[OPTS[new_k]] = opts_orig[old_k]
            new['answer'] = OPTS[int(np.where(perm == ans_idx)[0][0])]
            rows.append(new)
    aug = pd.DataFrame(rows)
    aug['prompt_n'] = aug['prompt'].map(normalize_text)
    for o in OPTS:
        aug[f'{o}_n'] = aug[o].map(normalize_text)
    return aug

# We DON'T concatenate aug data here — the option-shuffling Dataset handles
# position-bias augmentation on-the-fly. Hard-negative aug is opt-in below.
synth_df = hard_negative_augment(train, n_per_row=1) if CFG['RUN_SYNTH_GEN'] else pd.DataFrame()
print(f'Synthetic rows generated: {len(synth_df)}')


Synthetic rows generated: 0


## Metrics (MAP@3, Accuracy, Macro-F1)

In [7]:
def map_at_3(scores_2d, y_idx):
    scores_2d = np.asarray(scores_2d).reshape(-1, 5)
    y_idx = np.asarray(y_idx)
    order = (-scores_2d).argsort(axis=1)
    total = 0.0
    for i in range(len(scores_2d)):
        top3 = order[i, :3]
        if y_idx[i] in top3:
            total += 1.0 / (list(top3).index(y_idx[i]) + 1)
    return total / len(scores_2d)

def all_metrics(scores_2d, y_idx):
    scores_2d = np.asarray(scores_2d).reshape(-1, 5)
    y_idx = np.asarray(y_idx)
    pred_top1 = scores_2d.argmax(axis=1)
    return dict(
        map3 = map_at_3(scores_2d, y_idx),
        acc  = accuracy_score(y_idx, pred_top1),
        f1   = f1_score(y_idx, pred_top1, average='macro'),
    )

# Sanity check
_t = np.array([[0.1, 0.9, 0.05, 0.03, 0.02],
                [0.5, 0.2, 0.1, 0.1, 0.1]])
_y = np.array([1, 0])
print('Sanity:', all_metrics(_t, _y))  


Sanity: {'map3': 1.0, 'acc': 1.0, 'f1': 1.0}
